In [1]:
from datapreparation.data_preparation import Data_Preparation as dp
import pandas as pd
prep = dp()
X_train, X_val, X_test, y_train, y_val, y_test = prep.train_val_test_split_64_16_20_func(func= prep.prepare_data_pump_time_v2)
X_train

,hours_since_watering,soil_deficit_ratio,deficit_x_light,deficit_x_temp,deficit_x_air,et_approx
4105,273.0,0.768786,-3.230439,NaN,-2.429364,NaN
2730,182.0,0.091826,NaN,-0.247012,-0.604216,NaN
84024,83.0,0.562348,-2.303659,-0.073105,1.591445,0.714663
106895,133.0,0.182957,-0.054430,-0.064035,-0.934908,5.175905
8842,589.0,-0.017949,0.049188,0.052231,0.068026,0.283182
...,...,...,...,...,...,...
62835,219.0,0.341353,-0.661372,0.314045,-0.559820,2.006201
90812,2.0,0.220000,-0.397760,0.096800,-0.358600,4.092834
74102,31.0,0.216772,-0.281912,0.567943,0.351171,5.128716
75779,206.0,NaN,NaN,NaN,NaN,5.392630


In [2]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5, weights='distance')

X_train_imputed = imputer.fit_transform(X_train)

X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

In [3]:
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

X_val_imputed = pd.DataFrame(
    X_val_imputed,
    columns=X_val.columns,
    index=X_val.index
)
X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)
X_test_imputed

,hours_since_watering,soil_deficit_ratio,deficit_x_light,deficit_x_temp,deficit_x_air,et_approx
20928,135.0,0.394486,0.268251,-0.114401,-0.311644,5.400998
107419,117.0,0.305249,-0.047691,-0.423905,-0.911429,2.691243
41278,71.0,0.406634,-0.910453,-0.378170,-0.618084,0.000000
119396,32.0,0.398496,0.569850,0.187293,1.056015,5.750309
78091,75.0,0.199051,-0.707326,-0.007962,-0.494449,3.000649
...,...,...,...,...,...,...
31193,63.0,0.398496,-0.802173,-0.932481,-0.195263,0.000000
111514,331.0,0.172348,-0.571333,-0.475680,-0.243010,0.000000
66845,69.0,0.208696,-0.347947,0.252522,0.298435,1.801856
119041,6.0,0.360660,-0.373644,-0.490733,-0.567485,5.434638


In [4]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),

        "random_state": 42,
        "n_jobs": -1,
    }

    model = RandomForestRegressor(**params)

    model.fit(X_train_imputed, y_train)

    val_pred = model.predict(X_val_imputed)
    val_mae = mean_absolute_error(y_val, val_pred)

    return val_mae

study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    sampler= optuna.samplers.TPESampler(seed=42, multivariate=True,group=True,n_startup_trials=50)
)
study.optimize(objective, n_trials=60, show_progress_bar=True)

print("Best Trial:", study.best_trial.number)
print("Best MAE:", study.best_value)
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

best_params = study.best_params
best_params.update({
    "random_state": 42,
    "n_jobs": -1,
})

final_model = RandomForestRegressor(**best_params)
final_model.fit(X_train_imputed, y_train)

[I 2026-05-18 08:26:13,469] A new study created in memory with name: no-name-0b22c5db-7be8-4d01-b89a-7a63ac19c83c


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-05-18 08:26:18,292] Trial 0 finished with value: 0.8221838176281153 and parameters: {'n_estimators': 574, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.8221838176281153.
[I 2026-05-18 08:26:22,741] Trial 1 finished with value: 1.0015794300883971 and parameters: {'n_estimators': 908, 'max_depth': 4, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 0.8221838176281153.
[I 2026-05-18 08:26:30,847] Trial 2 finished with value: 0.8430339153582543 and parameters: {'n_estimators': 632, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True}. Best is trial 0 with value: 0.8221838176281153.
[I 2026-05-18 08:26:43,536] Trial 3 finished with value: 0.81372238141277 and parameters: {'n_estimators': 714, 'max_depth': 9, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",853
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples

In [5]:
test_pred = final_model.predict(X_test_imputed)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\nFinal Test MAE: {test_mae:.4f}")
print(f"Number of trees used: {final_model.n_estimators}")


Final Test MAE: 0.8112
Number of trees used: 853


In [6]:
import os
import joblib
# Delete old model before training
if os.path.exists("/models/pump_time/random_forest_model.ubj"):
    os.remove("/models/pump_time/random_forest_model.ubj")
    print("Old model deleted")

joblib.dump(final_model,"/workspace/models/pump_time/random_forest_model.ubj")
print("\nModel saved.")


Model saved.
